In [0]:
CREATE TABLE IF NOT EXISTS workspace.media_pipeline.dq_results (
  run_at      TIMESTAMP,
  check_name  STRING,
  layer       STRING,
  expected    DECIMAL(20,2),
  actual      DECIMAL(20,2),
  status      STRING
);

In [0]:
INSERT INTO workspace.media_pipeline.dq_results
SELECT current_timestamp(), check_name, layer, expected, actual,
       CASE WHEN expected = actual THEN 'PASS' ELSE 'FAIL' END
FROM (
  SELECT 'daily_rows_bronze_vs_silver' AS check_name, 'silver' AS layer,
         (SELECT COUNT(*) FROM workspace.media_pipeline.bronze_daily) AS expected,
         (SELECT COUNT(*) FROM workspace.media_pipeline.silver_daily) AS actual

  UNION ALL

  SELECT 'spend_silver_vs_fact', 'gold',
         (SELECT SUM(spend) FROM workspace.media_pipeline.silver_campaign_performance),
         (SELECT SUM(spend) FROM workspace.media_pipeline.gold_fact_campaign_performance)
 UNION ALL 
 SELECT 'social_rows_bronze_vs_silver' AS check_name, 'silver' AS layer,
         (SELECT COUNT(*) FROM workspace.media_pipeline.bronze_social) AS expected,
         (SELECT COUNT(*) FROM workspace.media_pipeline.silver_social) AS actual

  UNION ALL

  SELECT 'multichannel_rows_bronze_vs_silver', 'silver' AS layer,
         (SELECT COUNT(*) FROM workspace.media_pipeline.bronze_multichannel) AS expected, 
         (SELECT COUNT(*) FROM workspace.media_pipeline.silver_multichannel) AS actual
 UNION ALL
 SELECT 'orphan_campaigns' AS check_name,   -- what is checked
       'gold'             AS layer,         -- which layer
       0                  AS expected,      -- what should be true
       (SELECT COUNT(*)FROM workspace.media_pipeline.gold_fact_campaign_performance f
   LEFT JOIN workspace.media_pipeline.gold_dim_campaign c
     ON f.campaign_key = c.campaign_key
   WHERE c.campaign_key IS NULL )AS actual  -- what is actually found
   UNION ALL
   SELECT 'null_dates_non_social', 'silver',
         0,
         (SELECT COUNT_IF( report_date IS NULL AND platform <> 'social_platform')
          FROM workspace.media_pipeline.silver_campaign_performance)

) checks;

In [0]:
SELECT * FROM workspace.media_pipeline.dq_results
WHERE run_at = (SELECT MAX(run_at) FROM workspace.media_pipeline.dq_results)
ORDER BY check_name;